In [11]:
import requests
import pandas as pd
import re
import math

# API키 입력!
NAVER_CLIENT_ID = "--"
NAVER_CLIENT_SECRET = "--"
KAKAO_API_KEY = "--"

PNU_LAT = 35.2323
PNU_LNG = 129.0847

def get_naver_food():
    url = "https://openapi.naver.com/v1/search/local.json"
    headers = {
        "X-Naver-Client-Id": NAVER_CLIENT_ID,
        "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
    }
    params = {"query": "부산대 맛집", "display": 5, "sort": "comment"}
    res = requests.get(url, headers=headers, params=params)
    return res.json().get("items", [])

def get_naver_cafe():
    url = "https://openapi.naver.com/v1/search/local.json"
    headers = {
        "X-Naver-Client-Id": NAVER_CLIENT_ID,
        "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
    }
    params = {"query": "부산대 카페", "display": 5, "sort": "comment"}
    res = requests.get(url, headers=headers, params=params)
    return res.json().get("items", [])

def get_kakao_food(lat=PNU_LAT, lng=PNU_LNG, radius=500):
    url = "https://dapi.kakao.com/v2/local/search/category.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"category_group_code": "FD6", "x": lng, "y": lat, "radius": radius, "sort": "distance", "size": 15}
    res = requests.get(url, headers=headers, params=params)
    return res.json().get("documents", [])

def get_kakao_cafe(lat=PNU_LAT, lng=PNU_LNG, radius=500):
    url = "https://dapi.kakao.com/v2/local/search/category.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"category_group_code": "CE7", "x": lng, "y": lat, "radius": radius, "sort": "distance", "size": 15}
    res = requests.get(url, headers=headers, params=params)
    return res.json().get("documents", [])

def calc_walk_time(distance_m):
    if not distance_m:
        return "-"

    real_distance = int(distance_m) * 1.3
    minutes = real_distance / 67

    if minutes < 1:
        return "1분 이내"

    return f"약 {math.ceil(minutes)}분"

naver_food = get_naver_food()
naver_cafe = get_naver_cafe()
kakao_food = get_kakao_food()
kakao_cafe = get_kakao_cafe()


In [ ]:
food_naver_df = pd.DataFrame([{
    "이름": re.sub('<.*?>', '', r["title"]),
    "주소": r["roadAddress"],
    "카테고리": r["category"].split(">")[1].strip() if ">" in r["category"] else r["category"],
    "거리(m)": "-",
    "도보시간": "-",
    "출처": "네이버"
} for r in naver_food if "베이커리" not in r["category"] and "카페" not in r["category"]])

food_kakao_df = pd.DataFrame([{
    "이름": r["place_name"],
    "주소": r["road_address_name"],
    "카테고리": r["category_name"].split(">")[1].strip() if ">" in r["category_name"] else r["category_name"],
    "거리(m)": r["distance"],
    "도보시간": calc_walk_time(r["distance"]),
    "출처": "카카오"
} for r in kakao_food])

food_df = pd.concat([food_naver_df, food_kakao_df], ignore_index=True)
food_df

import ipywidgets as widgets
from IPython.display import display, clear_output

def get_main_category(category):
    category = category.replace(" ", "")
    if "한식" in category: return "한식"
    elif "일식" in category: return "일식"
    elif "중식" in category: return "중식"
    elif "양식" in category: return "양식"
    elif "치킨" in category: return "치킨"
    elif "피자" in category: return "피자"
    elif "분식" in category: return "분식"
    elif "아시아" in category or "베트남" in category or "인도" in category: return "아시아음식"
    elif "술집" in category or "호프" in category: return "술집"
    else: return "기타"

food_df["대분류"] = food_df["카테고리"].apply(get_main_category)

sort_dropdown1 = widgets.Dropdown(
    options=["거리순", "카테고리별"],
    description="밥집 정렬:",
    value="거리순"
)
output1 = widgets.Output()

def update_food(change):
    with output1:
        clear_output(wait=True)
        if sort_dropdown1.value == "거리순":
            result = food_df.copy()
            result["거리정렬"] = result["거리(m)"].apply(lambda x: int(x) if x != "-" else 9999)
            display(result.sort_values("거리정렬").drop(columns=["거리정렬"]).reset_index(drop=True))
        else:
            display(food_df.sort_values("대분류").reset_index(drop=True))

sort_dropdown1.observe(update_food, names="value")
display(sort_dropdown1, output1)
update_food(None)


Dropdown(description='밥집 정렬:', options=('거리순', '카테고리별'), value='거리순')

Output()

In [ ]:
cafe_naver_df = pd.DataFrame([{
    "이름": re.sub('<.*?>', '', r["title"]),
    "주소": r["roadAddress"],
    "카테고리": r["category"].split(">")[1].strip() if ">" in r["category"] else r["category"],
    "거리(m)": "-",
    "도보시간": "-",
    "출처": "네이버"
} for r in naver_cafe if "키즈" not in r["category"] and "보드" not in r["category"]])

cafe_kakao_df = pd.DataFrame([{
    "이름": r["place_name"],
    "주소": r["road_address_name"],
    "카테고리": r["category_name"].split(">")[1].strip() if ">" in r["category_name"] else r["category_name"],
    "거리(m)": r["distance"],
    "도보시간": calc_walk_time(r["distance"]),
    "출처": "카카오"
} for r in kakao_cafe if "키즈" not in r["category_name"] and "보드게임" not in r["category_name"]])

cafe_df = pd.concat([cafe_naver_df, cafe_kakao_df], ignore_index=True)
cafe_df

cafe_df["대분류"] = cafe_df["카테고리"].apply(get_main_category)

sort_dropdown2 = widgets.Dropdown(
    options=["거리순", "카테고리별"],
    description="카페 정렬:",
    value="거리순"
)
output2 = widgets.Output()

def update_cafe(change):
    with output2:
        clear_output(wait=True)
        if sort_dropdown2.value == "거리순":
            result = cafe_df.copy()
            result["거리정렬"] = result["거리(m)"].apply(lambda x: int(x) if x != "-" else 9999)
            display(result.sort_values("거리정렬").drop(columns=["거리정렬"]).reset_index(drop=True))
        else:
            display(cafe_df.sort_values("대분류").reset_index(drop=True))

sort_dropdown2.observe(update_cafe, names="value")
display(sort_dropdown2, output2)
update_cafe(None)


Dropdown(description='카페 정렬:', options=('거리순', '카테고리별'), value='거리순')

Output()